In [ ]:
# 0: ancillary qubit for block encoding
# 1,2: wire_i for block encoding
# 3,4: 00 for psi_1, 01 for psi_2, 10 for psi_10
# 5,6: wire_j

In [ ]:
from qiskit import QuantumCircuit
from numpy import sqrt, array


def circuit_init():
    qc = QuantumCircuit(7)
    return qc
    
def PREP(qc):

    desired_vector = [
        1/sqrt(3), 0, 0, 0, 
        1/(4*sqrt(3)), 1/4, 1/4, sqrt(3)/4, 
        1/(4*sqrt(3)), -1/4, -1/4, sqrt(3)/4, 
        0, 0, 0, 0
    ]

    # Initialize the quantum circuit with the normalized desired vector
    qc.initialize(desired_vector, [6,5,4,3])
    
    return qc

In [ ]:
from pennylane.templates.state_preparations.mottonen import compute_theta, gray_code
import numpy as np
from qiskit import transpile
from qiskit_aer import Aer, AerSimulator

A = np.array ([
    [1.70710678, 0, 0, -0.29289321],
    [0, 1, 1, 0],
    [0, 1, 1, 0],
    [-0.29289321, 0, 0, 1.70710678]
])

ancilla_wire = [0]
wires_i = [1, 2]
wires_j = [5, 6]


s = int(np.log2(A.shape[0]))

A = A / np.max(np.abs(A))
thetas = np.arccos(A).flatten()
thetas = compute_theta(thetas)

code = gray_code(2 * np.log2(len(A)))
n_selections = len(code)
control_order = [int(np.log2(int(code[i], 2) ^ int(code[(i + 1) % n_selections], 2))) for i in range(n_selections)]

tolerance = 0.01
def UT(qc, thetas, control_wires, ancilla_wire):
    ancilla = ancilla_wire[0]  
    nots = []

    for theta, control_index in zip(thetas, control_wires):
        if abs(2 * theta) > tolerance:
            for c_wire in nots:
                qc.cx(c_wire, ancilla)
            qc.ry(2 * theta, ancilla)
            nots = []

        if control_index in nots:
            nots.remove(control_index)  
        else:
            nots.append(control_index)  

    for c_wire in nots:
        qc.cx(c_wire, ancilla)
        
def UB(qc, wires_i, wires_j):
    for w_i, w_j in zip(wires_i, wires_j):

        qc.swap(w_i, w_j)

def HN(qc, input_wires):
    for w in input_wires:
        qc.h(w)
        
input_wires = ancilla_wire + wires_i

def get_control_qubit(control_order, wires_i, wires_j):
    n = len(wires_i)  # Same as len(wires_j)
    control_qubits = []

    for control_value in control_order:
        # Control values between 0 and n-1 correspond to wires_j (reverse order)
        if control_value < n:
            control_qubits.append(wires_j[-(control_value + 1)])  # Reverse index for wires_j
        # Control values between n and 2n-1 correspond to wires_i (reverse order)
        elif control_value < 2 * n:
            control_qubits.append(wires_i[-(control_value - n + 1)])  # Reverse index for wires_i

    return control_qubits

control_wires = get_control_qubit(control_order, wires_i, wires_j)

def Block_Encoding(qc):
    HN(qc, wires_i)
    qc.barrier()  
    UT(qc, thetas, control_wires, ancilla_wire)
    qc.barrier()   
    UB(qc, wires_i, wires_j)
    HN(qc, wires_i)
    qc.barrier()
    
    return qc

In [ ]:
from qiskit.circuit.library.standard_gates import U3Gate, RZGate

def UU(qc, wires, params):
    qc.append(U3Gate(params[0],params[1], params[2]),[wires[0]])
    qc.append(U3Gate(params[3],params[4], params[5]),[wires[1]])

def RR_Z(qc, wires, params):
    qc.append(RZGate(params[0]),[wires[0]])
    qc.append(RZGate(params[1]),[wires[1]])

def Ansatz(qc, wires, params):
    UU(qc,wires[0:2],params[0:6])
    qc.cx(wires[1],wires[0])
    RR_Z(qc,wires[0:2],params[6:8])
    qc.cx(wires[0],wires[1])
    qc.ry(params[8],wires[1])
    qc.cx(wires[1],wires[0])
    UU(qc,wires[0:2],params[9:15])
    qc.measure_all()
    return qc

In [ ]:
from numpy.random import rand
num_params= 15
params_instant = rand(num_params)
num_shots=10000

qc = circuit_init()
qc = PREP(qc)
qc.barrier()
qc = Block_Encoding(qc)
Ansatz(qc, wires_j, params_instant)

qc.draw(output='mpl', style = 'clifford') 
#qc_reversed=qc.reverse_bits()
#qc_reversed.draw(output='mpl',style = 'clifford') 

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile
from scipy.optimize import minimize

import numpy as np
import os
import time


# ============================================================
# Simulator
# ============================================================

aersim = AerSimulator()


# ============================================================
# Settings
# ============================================================

num_times = 100
num_shots = 10000
optimizer_name = "COBYLA"
tol_value = 1e-8
maxiter = 1000
BASE_SEED = 150

report_filename = (
    "DT_Naimark_01_MCM_report.txt"
)


# ============================================================
# High-Shot Evaluation Settings
# ============================================================

evaluation_num_times = 20
evaluation_num_shots = 100000

EVALUATION_BASE_SEED = 10000

evaluation_seeds = np.array(
    [
        EVALUATION_BASE_SEED + i
        for i in range(evaluation_num_times)
    ],
    dtype=int
)


# ============================================================
# Storage
# ============================================================

final_optimized_values = []

initial_params_all_runs = []
optimal_params_all_runs = []

optimization_success_all_runs = []
optimization_status_all_runs = []
optimization_message_all_runs = []
optimization_nfev_all_runs = []
reached_maxiter_all_runs = []

# ------------------------------------------------------------
# Runtime Storage
# ------------------------------------------------------------

optimization_wall_times = []


# ------------------------------------------------------------
# High-Shot Evaluation Storage
# ------------------------------------------------------------

evaluation_values_all_runs = []

evaluation_mean_all_runs = []
evaluation_std_all_runs = []
evaluation_min_all_runs = []
evaluation_max_all_runs = []


# ============================================================
# Objective Function
# ============================================================

def objective_function(
    params,
    run_seed,
    shots=None,
    print_result=True
):

    if shots is None:
        shots = num_shots


    # --------------------------------------------------------
    # Construct circuit
    # --------------------------------------------------------

    qc = circuit_init()

    PREP(qc)

    Block_Encoding(qc)

    Ansatz(
        qc,
        wires_j,
        params=params
    )

    # Add measurement if Ansatz itself does NOT measure
    qc.measure_all()

    # Reverse qubit/bit ordering
    qc_reverse = qc.reverse_bits()

    # --------------------------------------------------------
    # Transpile
    # --------------------------------------------------------

    t_qc = transpile(
        qc_reverse,
        aersim,
        seed_transpiler=run_seed
    )

    # --------------------------------------------------------
    # Run simulation
    # --------------------------------------------------------

    job = aersim.run(
        t_qc,
        shots=shots,
        seed_simulator=run_seed
    )

    result = job.result()
    counts = result.get_counts()

    # --------------------------------------------------------
    # Conditional probability counts
    # --------------------------------------------------------

    target_counts = 0
    total_counts = 0

    # Choose target state
    # psi = "00"
    psi = "01"
    # psi = "10"

    for outcome, count in counts.items():
        if outcome[0:5] == '000'+psi:  # Check if b0b1b2b3b4 = 00000
            total_counts += count
            if outcome[5:7] == psi:  # Check if b5b6 = 00
                target_counts += count


    
    # --------------------------------------------------------
    # Calculate conditional probability
    # --------------------------------------------------------

    if total_counts > 0:
        conditional_prob = target_counts / total_counts
    else:
        conditional_prob = 0.0

    if print_result:

        print(
            f"Conditional probability = "
            f"{conditional_prob:.8f}"
        )

    # --------------------------------------------------------
    # Objective for COBYLA
    # --------------------------------------------------------

    if conditional_prob > 0:
        return 1.0 / conditional_prob
    else:
        return 1e10


# ============================================================
# Run Optimization
# ============================================================

# Measures the complete workflow:
# optimization + final evaluations + high-shot evaluations + overhead.
total_wall_start = time.perf_counter()

for run in range(num_times):


    # ========================================================
    # Optimization Seed
    # ========================================================

    run_seed = (
        BASE_SEED + run
    )

    rng = np.random.default_rng(
        run_seed
    )


    print(
        "\n"
        + "=" * 80
    )
    print(
        f"Optimization Run "
        f"{run + 1}/{num_times}"
    )


    print(
        f"Optimization Seed : "
        f"{run_seed}"
    )


    print(
        "=" * 80
    )

    # ========================================================
    # Generate Random Initial Parameters
    # ========================================================

    initial_params = rng.random(num_params)

    # Store initial parameters
    initial_params_all_runs.append(
        initial_params.copy()
    )

    print("\nInitial Parameters:")

    print(
        np.array2string(
            initial_params,
            precision=10,
            separator=", "
        )
    )

    # ========================================================
    # Run COBYLA Optimization
    # ========================================================

    print(
        "\nStarting COBYLA optimization..."
    )


    optimization_start = time.perf_counter()

    result = minimize(
        fun=lambda params: objective_function(
            params,
            run_seed,
            shots=num_shots,
            print_result=True
        ),
        x0=initial_params,
        method=optimizer_name,
        options={
            "maxiter": maxiter,
            "disp": True,
            "tol": tol_value
        }
    )

    optimization_end = time.perf_counter()

    run_optimization_wall_time = (
        optimization_end
        - optimization_start
    )

    optimization_wall_times.append(
        run_optimization_wall_time
    )

    print(
        f"Optimization Wall-Clock Time : "
        f"{run_optimization_wall_time:.6f} s"
    )


    # ========================================================
    # Optimization Status
    # --------------------------------------------------------

    optimization_success = bool(
        result.success
    )

    optimization_status = getattr(
        result,
        "status",
        None
    )

    optimization_message = str(
        getattr(
            result,
            "message",
            "No message returned"
        )
    )

    optimization_nfev = getattr(
        result,
        "nfev",
        None
    )

    # ========================================================
    # Detect Maximum Function Evaluations
    # ========================================================

    message_lower = optimization_message.lower()

    reached_maxiter = (
        "maximum number" in message_lower
        or "maximum function" in message_lower
        or "maxiter" in message_lower
        or "maxfun" in message_lower
        or "maximum iterations" in message_lower
        or "maximum evaluations" in message_lower
    )

    if optimization_nfev is not None:
        if optimization_nfev >= maxiter:
            reached_maxiter = True

    # ========================================================
    # Store Optimization Status
    # ========================================================

    optimization_success_all_runs.append(
        optimization_success
    )

    optimization_status_all_runs.append(
        optimization_status
    )

    optimization_message_all_runs.append(
        optimization_message
    )

    optimization_nfev_all_runs.append(
        optimization_nfev
    )

    reached_maxiter_all_runs.append(
        reached_maxiter
    )

    # ========================================================
    # Optimal Parameters
    # ========================================================

    optimal_params = result.x.copy()

    # Store optimal parameters
    optimal_params_all_runs.append(
        optimal_params.copy()
    )

    # ========================================================
    # Final Optimization-Shot Evaluation
    # ========================================================

    final_objective = objective_function(
        optimal_params,
        run_seed,
        shots=num_shots,
        print_result=False
    )

    if final_objective >= 1e10:
        final_value = 0.0
    else:
        final_value = 1.0 / final_objective

    final_optimized_values.append(
        final_value
    )

    # ========================================================
    # High-Shot Evaluation
    # ========================================================

    print(
        "\n"
        + "-" * 80
    )

    print(
        "HIGH-SHOT EVALUATION OF "
        "OPTIMIZED PARAMETERS"
    )

    print(
        "-" * 80
    )

    print(
        f"Optimization Shots     : "
        f"{num_shots}"
    )

    print(
        f"Evaluation Shots       : "
        f"{evaluation_num_shots}"
    )

    print(
        f"Number of Evaluations : "
        f"{evaluation_num_times}"
    )

    print(
        "\nEvaluation Results:"
    )

    evaluation_values = []

    for eval_index, eval_seed in enumerate(
        evaluation_seeds
    ):

        eval_objective = objective_function(
            optimal_params,
            int(eval_seed),
            shots=evaluation_num_shots,
            print_result=False
        )

        if eval_objective >= 1e10:
            eval_value = 0.0
        else:
            eval_value = 1.0 / eval_objective

        evaluation_values.append(
            eval_value
        )

        print(
            f"Evaluation "
            f"{eval_index + 1:2d}/"
            f"{evaluation_num_times:2d}"
            f" | Value = "
            f"{eval_value:.8f}"
        )

    # ========================================================
    # Evaluation Statistics
    # ========================================================

    evaluation_values = np.array(
        evaluation_values,
        dtype=float
    )

    evaluation_mean = np.mean(
        evaluation_values
    )

    evaluation_std = np.std(
        evaluation_values
    )

    evaluation_min = np.min(
        evaluation_values
    )

    evaluation_max = np.max(
        evaluation_values
    )

    # ========================================================
    # Store Evaluation Results
    # ========================================================

    evaluation_values_all_runs.append(
        evaluation_values.copy()
    )

    evaluation_mean_all_runs.append(
        evaluation_mean
    )

    evaluation_std_all_runs.append(
        evaluation_std
    )

    evaluation_min_all_runs.append(
        evaluation_min
    )

    evaluation_max_all_runs.append(
        evaluation_max
    )


    # ========================================================
    # Individual Run Report
    # ========================================================

    print(
        "\n"
        + "=" * 80
    )

    print(
        f"RUN {run + 1} RESULT"
    )

    print(
        "=" * 80
    )

    print(
        f"Optimization Success : "
        f"{optimization_success}"
    )

    print(
        f"Reached MaxIter      : "
        f"{reached_maxiter}"
    )

    print(
        f"Status               : "
        f"{optimization_status}"
    )

    print(
        f"Message              : "
        f"{optimization_message}"
    )

    print(
        f"Function Evaluations : "
        f"{optimization_nfev}"
    )

    print(
        f"Optimization Time    : "
        f"{run_optimization_wall_time:.6f} s"
    )

    print(
        f"Final Objective Value: "
        f"{result.fun}"
    )

    print(
        "\n"
        + "-" * 80
    )

    print(
        "OPTIMIZED AND HIGH-SHOT "
        "EVALUATED VALUES"
    )

    print(
        "-" * 80
    )

    print(
        f"Final Optimized Value : "
        f"{final_value:.8f}"
    )

    print(
        f"Evaluated Mean        : "
        f"{evaluation_mean:.8f}"
    )

    print(
        f"Evaluated Std         : "
        f"{evaluation_std:.8f}"
    )

    print(
        f"Evaluated Min         : "
        f"{evaluation_min:.8f}"
    )

    print(
        f"Evaluated Max         : "
        f"{evaluation_max:.8f}"
    )

    print(
        "\nAll Evaluated Values:"
    )

    print(
        np.array2string(
            evaluation_values,
            precision=8,
            separator=", "
        )
    )

    print(
        "\nInitial Parameters:"
    )

    print(
        np.array2string(
            initial_params,
            precision=10,
            separator=", "
        )
    )

    print(
        "\nOptimal Parameters:"
    )

    print(
        np.array2string(
            optimal_params,
            precision=10,
            separator=", "
        )
    )

    print(
        "=" * 80
    )


# ============================================================
# Runtime Statistics
# ============================================================

total_wall_end = time.perf_counter()

total_wall_time = (
    total_wall_end
    - total_wall_start
)

optimization_wall_times = np.array(
    optimization_wall_times,
    dtype=float
)

total_optimization_wall_time = np.sum(
    optimization_wall_times
)

mean_optimization_wall_time = np.mean(
    optimization_wall_times
)

std_optimization_wall_time = np.std(
    optimization_wall_times
)


# ============================================================
# Convert Results to NumPy Arrays
# ============================================================

final_optimized_values = np.array(
    final_optimized_values
)

initial_params_all_runs = np.array(
    initial_params_all_runs
)

optimal_params_all_runs = np.array(
    optimal_params_all_runs
)

evaluation_values_all_runs = np.array(
    evaluation_values_all_runs,
    dtype=float
)

evaluation_mean_all_runs = np.array(
    evaluation_mean_all_runs,
    dtype=float
)

evaluation_std_all_runs = np.array(
    evaluation_std_all_runs,
    dtype=float
)

evaluation_min_all_runs = np.array(
    evaluation_min_all_runs,
    dtype=float
)

evaluation_max_all_runs = np.array(
    evaluation_max_all_runs,
    dtype=float
)


# ============================================================
# Original Optimization Statistics
# ============================================================

maximum_value = np.max(
    final_optimized_values
)

mean_value = np.mean(
    final_optimized_values
)

std_value = np.std(
    final_optimized_values
)

# ------------------------------------------------------------
# Function Evaluation Statistics
# ------------------------------------------------------------

nfev_values = np.array(
    [
        nfev
        for nfev in optimization_nfev_all_runs
        if nfev is not None
    ],
    dtype=float
)

if len(nfev_values) > 0:
    mean_nfev = np.mean(
        nfev_values
    )

    std_nfev = np.std(
        nfev_values
    )
else:
    mean_nfev = np.nan
    std_nfev = np.nan

maximum_run_index = np.argmax(
    final_optimized_values
)

maximum_run = maximum_run_index + 1


# ============================================================
# High-Shot Evaluation Statistics
# ============================================================

maximum_evaluated_mean = np.max(
    evaluation_mean_all_runs
)

maximum_evaluated_mean_run_index = np.argmax(
    evaluation_mean_all_runs
)

maximum_evaluated_mean_run = (
    maximum_evaluated_mean_run_index
    + 1
)

mean_of_evaluated_means = np.mean(
    evaluation_mean_all_runs
)

std_of_evaluated_means = np.std(
    evaluation_mean_all_runs
)


# ============================================================
# Optimization Success Statistics
# ============================================================

num_successful_runs = sum(
    optimization_success_all_runs
)

num_failed_runs = (
    num_times
    - num_successful_runs
)

num_reached_maxiter = sum(
    reached_maxiter_all_runs
)


# ============================================================
# Best Run Parameters
# ============================================================

best_initial_params = initial_params_all_runs[
    maximum_run_index
]

best_optimal_params = optimal_params_all_runs[
    maximum_run_index
]


# ============================================================
# Best Run According to High-Shot Evaluated Mean
# ============================================================

best_evaluated_initial_params = (
    initial_params_all_runs[
        maximum_evaluated_mean_run_index
    ]
)

best_evaluated_optimal_params = (
    optimal_params_all_runs[
        maximum_evaluated_mean_run_index
    ]
)

best_evaluated_values = (
    evaluation_values_all_runs[
        maximum_evaluated_mean_run_index
    ]
)


# ============================================================
# Print Final Report
# ============================================================

print("\n")
print("=" * 100)
print(
    "FINAL OPTIMIZATION AND "
    "HIGH-SHOT EVALUATION REPORT"
)
print("=" * 100)

print(f"Optimizer               : {optimizer_name}")
print(f"Tolerance               : {tol_value}")
print(f"Max Iteration           : {maxiter}")
print(f"Number of Runs          : {num_times}")
print(f"Optimization Shots      : {num_shots}")
print(f"Optimization Base Seed  : {BASE_SEED}")
print(f"Evaluation Shots        : {evaluation_num_shots}")
print(f"Evaluations Per Run     : {evaluation_num_times}")
print(f"Evaluation Base Seed    : {EVALUATION_BASE_SEED}")
print(
    "Evaluation Seeds        : "
    + np.array2string(evaluation_seeds, separator=", ")
)
print("=" * 100)

# ============================================================
# Results for Every Run
# ============================================================

for i in range(num_times):
    print(f"\nRun {i + 1:2d}")
    print("-" * 100)
    print(f"Optimization Seed : {BASE_SEED + i}")
    print(f"Optimized Value   : {final_optimized_values[i]:.8f}")
    print(f"Evaluated Mean    : {evaluation_mean_all_runs[i]:.8f}")
    print(f"Evaluated Std     : {evaluation_std_all_runs[i]:.8f}")
    print(f"Evaluated Min     : {evaluation_min_all_runs[i]:.8f}")
    print(f"Evaluated Max     : {evaluation_max_all_runs[i]:.8f}")
    print(f"Success           : {optimization_success_all_runs[i]}")
    print(f"Reached MaxIter   : {reached_maxiter_all_runs[i]}")
    print(f"Status            : {optimization_status_all_runs[i]}")
    print(f"Message           : {optimization_message_all_runs[i]}")
    print(f"Function Evals    : {optimization_nfev_all_runs[i]}")
    print(f"Optimization Time : {optimization_wall_times[i]:.6f} s")
    print("\nEvaluated Values:")
    print(np.array2string(evaluation_values_all_runs[i], precision=8, separator=", "))
    print("\nInitial Parameters:")
    print(np.array2string(initial_params_all_runs[i], precision=10, separator=", "))
    print("\nOptimal Parameters:")
    print(np.array2string(optimal_params_all_runs[i], precision=10, separator=", "))
    print("-" * 100)

# ============================================================
# Summary
# ============================================================

print("\nSUMMARY")
print("=" * 100)
print("\nOriginal Optimization-Shot Results:")
print(f"Maximum Optimized Value : {maximum_value:.8f}")
print(f"Maximum Run             : {maximum_run}")
print(f"Mean Optimized Value    : {mean_value:.8f}")
print(f"Std Optimized Value     : {std_value:.8f}")
print("\nHigh-Shot Evaluation Results:")
print(f"Best Evaluated Mean     : {maximum_evaluated_mean:.8f}")
print(f"Best Evaluated Mean Run : {maximum_evaluated_mean_run}")
print(f"Mean of Evaluated Means : {mean_of_evaluated_means:.8f}")
print(f"Std of Evaluated Means  : {std_of_evaluated_means:.8f}")
print("\nOptimizer Statistics:")
print(f"Mean Function Evals     : {mean_nfev:.2f}")
print(f"Std Function Evals      : {std_nfev:.2f}")
print(f"Successful Runs         : {num_successful_runs}/{num_times}")
print(f"Failed Runs             : {num_failed_runs}/{num_times}")
print(f"Reached MaxIter         : {num_reached_maxiter}/{num_times}")
print("\nComputational Runtime Statistics:")
print(f"Total Wall-Clock Time          : {total_wall_time:.6f} s")
print(f"Total Optimization Time        : {total_optimization_wall_time:.6f} s")
print(f"Time per Complete Opt. Run     : {mean_optimization_wall_time:.6f} s")
print(f"Std Optimization Time per Run  : {std_optimization_wall_time:.6f} s")

# ============================================================
# Best Run According to Original Optimized Value
# ============================================================

print("\n" + "=" * 100)
print("BEST RUN ACCORDING TO ORIGINAL OPTIMIZED VALUE")
print("=" * 100)
print(f"Run             : {maximum_run}")
print(f"Optimized Value : {final_optimized_values[maximum_run_index]:.8f}")
print(f"Evaluated Mean  : {evaluation_mean_all_runs[maximum_run_index]:.8f}")
print(f"Evaluated Std   : {evaluation_std_all_runs[maximum_run_index]:.8f}")
print(f"Evaluated Min   : {evaluation_min_all_runs[maximum_run_index]:.8f}")
print(f"Evaluated Max   : {evaluation_max_all_runs[maximum_run_index]:.8f}")
print("\nEvaluated Values:")
print(np.array2string(evaluation_values_all_runs[maximum_run_index], precision=8, separator=", "))
print("\nInitial Parameters:")
print(np.array2string(best_initial_params, precision=10, separator=", "))
print("\nOptimal Parameters:")
print(np.array2string(best_optimal_params, precision=10, separator=", "))

# ============================================================
# Best Run According to High-Shot Evaluated Mean
# ============================================================

print("\n" + "=" * 100)
print("BEST RUN ACCORDING TO HIGH-SHOT EVALUATED MEAN")
print("=" * 100)
print(f"Run             : {maximum_evaluated_mean_run}")
print(f"Optimized Value : {final_optimized_values[maximum_evaluated_mean_run_index]:.8f}")
print(f"Evaluated Mean  : {evaluation_mean_all_runs[maximum_evaluated_mean_run_index]:.8f}")
print(f"Evaluated Std   : {evaluation_std_all_runs[maximum_evaluated_mean_run_index]:.8f}")
print(f"Evaluated Min   : {evaluation_min_all_runs[maximum_evaluated_mean_run_index]:.8f}")
print(f"Evaluated Max   : {evaluation_max_all_runs[maximum_evaluated_mean_run_index]:.8f}")
print("\nEvaluated Values:")
print(np.array2string(best_evaluated_values, precision=8, separator=", "))
print("\nInitial Parameters:")
print(np.array2string(best_evaluated_initial_params, precision=10, separator=", "))
print("\nOptimal Parameters:")
print(np.array2string(best_evaluated_optimal_params, precision=10, separator=", "))
print("=" * 100)


# ============================================================
# Save Report to TXT
# ============================================================

with open(report_filename, "w") as f:

    f.write("=" * 100 + "\n")
    f.write("FINAL OPTIMIZATION AND HIGH-SHOT EVALUATION REPORT\n")
    f.write("=" * 100 + "\n\n")

    f.write(f"Optimizer               : {optimizer_name}\n")
    f.write(f"Tolerance               : {tol_value}\n")
    f.write(f"Max Iteration           : {maxiter}\n")
    f.write(f"Number of Runs          : {num_times}\n")
    f.write(f"Optimization Shots      : {num_shots}\n")
    f.write(f"Optimization Base Seed  : {BASE_SEED}\n")
    f.write(f"Evaluation Shots        : {evaluation_num_shots}\n")
    f.write(f"Evaluations Per Run     : {evaluation_num_times}\n")
    f.write(f"Evaluation Base Seed    : {EVALUATION_BASE_SEED}\n")
    f.write("Evaluation Seeds        : ")
    f.write(np.array2string(evaluation_seeds, separator=", "))
    f.write("\n\n")

    f.write("=" * 100 + "\n")
    f.write("RESULTS FOR EACH RUN\n")
    f.write("=" * 100 + "\n")

    for i in range(num_times):
        f.write(f"\nRun {i + 1:2d}\n")
        f.write("-" * 100 + "\n")
        f.write(f"Optimization Seed : {BASE_SEED + i}\n")
        f.write(f"Optimized Value   : {final_optimized_values[i]:.8f}\n")
        f.write(f"Evaluated Mean    : {evaluation_mean_all_runs[i]:.8f}\n")
        f.write(f"Evaluated Std     : {evaluation_std_all_runs[i]:.8f}\n")
        f.write(f"Evaluated Min     : {evaluation_min_all_runs[i]:.8f}\n")
        f.write(f"Evaluated Max     : {evaluation_max_all_runs[i]:.8f}\n")
        f.write(f"Success           : {optimization_success_all_runs[i]}\n")
        f.write(f"Reached MaxIter   : {reached_maxiter_all_runs[i]}\n")
        f.write(f"Status            : {optimization_status_all_runs[i]}\n")
        f.write(f"Message           : {optimization_message_all_runs[i]}\n")
        f.write(f"Function Evals    : {optimization_nfev_all_runs[i]}\n")
        f.write(f"Optimization Time : {optimization_wall_times[i]:.6f} s\n")
        f.write("\nEvaluated Values:\n")
        f.write(np.array2string(evaluation_values_all_runs[i], precision=8, separator=", "))
        f.write("\n\nInitial Parameters:\n")
        f.write(np.array2string(initial_params_all_runs[i], precision=10, separator=", "))
        f.write("\n\nOptimal Parameters:\n")
        f.write(np.array2string(optimal_params_all_runs[i], precision=10, separator=", "))
        f.write("\n" + "-" * 100 + "\n")

    f.write("\n" + "=" * 100 + "\n")
    f.write("SUMMARY\n")
    f.write("=" * 100 + "\n")
    f.write("\nOriginal Optimization-Shot Results:\n")
    f.write(f"Maximum Optimized Value : {maximum_value:.8f}\n")
    f.write(f"Maximum Run             : {maximum_run}\n")
    f.write(f"Mean Optimized Value    : {mean_value:.8f}\n")
    f.write(f"Std Optimized Value     : {std_value:.8f}\n")
    f.write("\nHigh-Shot Evaluation Results:\n")
    f.write(f"Best Evaluated Mean     : {maximum_evaluated_mean:.8f}\n")
    f.write(f"Best Evaluated Mean Run : {maximum_evaluated_mean_run}\n")
    f.write(f"Mean of Evaluated Means : {mean_of_evaluated_means:.8f}\n")
    f.write(f"Std of Evaluated Means  : {std_of_evaluated_means:.8f}\n")
    f.write("\nOptimizer Statistics:\n")
    f.write(f"Mean Function Evals     : {mean_nfev:.2f}\n")
    f.write(f"Std Function Evals      : {std_nfev:.2f}\n")
    f.write(f"Successful Runs         : {num_successful_runs}/{num_times}\n")
    f.write(f"Failed Runs             : {num_failed_runs}/{num_times}\n")
    f.write(f"Reached MaxIter         : {num_reached_maxiter}/{num_times}\n")
    f.write("\nComputational Runtime Statistics:\n")
    f.write(f"Total Wall-Clock Time          : {total_wall_time:.6f} s\n")
    f.write(f"Total Optimization Time        : {total_optimization_wall_time:.6f} s\n")
    f.write(f"Time per Complete Opt. Run     : {mean_optimization_wall_time:.6f} s\n")
    f.write(f"Std Optimization Time per Run  : {std_optimization_wall_time:.6f} s\n")

    f.write("\n" + "=" * 100 + "\n")
    f.write("BEST RUN ACCORDING TO ORIGINAL OPTIMIZED VALUE\n")
    f.write("=" * 100 + "\n")
    f.write(f"Run             : {maximum_run}\n")
    f.write(f"Optimized Value : {final_optimized_values[maximum_run_index]:.8f}\n")
    f.write(f"Evaluated Mean  : {evaluation_mean_all_runs[maximum_run_index]:.8f}\n")
    f.write(f"Evaluated Std   : {evaluation_std_all_runs[maximum_run_index]:.8f}\n")
    f.write(f"Evaluated Min   : {evaluation_min_all_runs[maximum_run_index]:.8f}\n")
    f.write(f"Evaluated Max   : {evaluation_max_all_runs[maximum_run_index]:.8f}\n")
    f.write("\nEvaluated Values:\n")
    f.write(np.array2string(evaluation_values_all_runs[maximum_run_index], precision=8, separator=", "))
    f.write("\n\nInitial Parameters:\n")
    f.write(np.array2string(best_initial_params, precision=10, separator=", "))
    f.write("\n\nOptimal Parameters:\n")
    f.write(np.array2string(best_optimal_params, precision=10, separator=", "))

    f.write("\n\n" + "=" * 100 + "\n")
    f.write("BEST RUN ACCORDING TO HIGH-SHOT EVALUATED MEAN\n")
    f.write("=" * 100 + "\n")
    f.write(f"Run             : {maximum_evaluated_mean_run}\n")
    f.write(f"Optimized Value : {final_optimized_values[maximum_evaluated_mean_run_index]:.8f}\n")
    f.write(f"Evaluated Mean  : {evaluation_mean_all_runs[maximum_evaluated_mean_run_index]:.8f}\n")
    f.write(f"Evaluated Std   : {evaluation_std_all_runs[maximum_evaluated_mean_run_index]:.8f}\n")
    f.write(f"Evaluated Min   : {evaluation_min_all_runs[maximum_evaluated_mean_run_index]:.8f}\n")
    f.write(f"Evaluated Max   : {evaluation_max_all_runs[maximum_evaluated_mean_run_index]:.8f}\n")
    f.write("\nEvaluated Values:\n")
    f.write(np.array2string(best_evaluated_values, precision=8, separator=", "))
    f.write("\n\nInitial Parameters:\n")
    f.write(np.array2string(best_evaluated_initial_params, precision=10, separator=", "))
    f.write("\n\nOptimal Parameters:\n")
    f.write(np.array2string(best_evaluated_optimal_params, precision=10, separator=", "))
    f.write("\n" + "=" * 100 + "\n")


# ============================================================
# Report Location
# ============================================================

print(
    "\nReport saved to:"
)

print(
    os.path.abspath(
        report_filename
    )
)